# LBot V7 - Seq2Seq Encoder-Decoder Training (Robust Edition)

**Arquitetura:** Seq2Seq com Bidirectional GRU + Bahdanau Attention (mesma da V6)

| Parâmetro | V6 (Seq2Seq) | V7 (Seq2Seq Robust) |
|-----------|--------------|---------------------|
| Dataset | 80k exemplos, só cm | 220k exemplos, multi-unidade + augmentation |
| Distâncias | 1-100 cm | 1-999 cm (clamping > 999) |
| Ângulos | 30,45,60,90,120,135,150,180 | 1-360 (todos os inteiros) |
| Max input len | 200 | 250 (números por extenso são mais longos) |
| Augmentation | Nenhuma | Sem acentos, abreviações, typos, números por extenso, informal |
| Unidades no dataset | Só centímetros | Só centímetros (conversão no preprocessing) |
| Vocabulário informal | Não | Sim (vai, roda, dobra, pra frente, reto, etc.) |
| Preprocessing (inference) | Só lowercase | Pipeline completo: unidades→cm, clamping, acentos, pontuação, abreviações |
| Batch size | 64 | 128 |
| Max epochs | 100 | 150 |
| Early stopping patience | 10 | 15 |

**LBML V4 Format (unchanged):**
- Deslocamento: `D<valor_cm><direção>;` (ex: `D40F;`, `D350F;`, `D999F;`)
- Rotação: `R<ângulo><direção>;` (ex: `R90R;`, `R270L;`, `R360R;`)
- Composto: `D40F;R90R;D200L;`

## 1. Instalação e Imports

In [ ]:
!pip install torch numpy matplotlib tqdm -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import random
import re
import os
import math
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 2. Tokens Especiais e Vocabulário

In [ ]:
# Special tokens
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'

SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3


class Vocabulary:
    """Character-level vocabulary with special tokens."""

    def __init__(self, name: str = 'vocab'):
        self.name = name
        self.stoi: Dict[str, int] = {}
        self.itos: Dict[int, str] = {}
        self.size = 0
        for token in SPECIAL_TOKENS:
            self._add_token(token)

    def _add_token(self, token: str):
        if token not in self.stoi:
            idx = self.size
            self.stoi[token] = idx
            self.itos[idx] = token
            self.size += 1

    def build_from_texts(self, texts: List[str]):
        chars = set()
        for text in texts:
            chars.update(text)
        for char in sorted(chars):
            self._add_token(char)

    def encode(self, text: str) -> List[int]:
        return [self.stoi.get(c, UNK_IDX) for c in text]

    def decode(self, indices: List[int], strip_special: bool = True) -> str:
        tokens = []
        for idx in indices:
            token = self.itos.get(idx, UNK_TOKEN)
            if strip_special and token in SPECIAL_TOKENS:
                if token == EOS_TOKEN:
                    break
                continue
            tokens.append(token)
        return ''.join(tokens)

    def __len__(self):
        return self.size


print('✅ Vocabulary class defined')

## 3. Carregar Dataset

In [ ]:
# Upload dataset file (run generate_dataset_v7.py first)
from google.colab import files

print('📁 Upload lbot_dataset_v7.txt')
uploaded = files.upload()
dataset_file = list(uploaded.keys())[0]
print(f'✅ Uploaded: {dataset_file}')

In [ ]:
def load_dataset(filename: str) -> List[Tuple[str, str]]:
    """Load dataset from file. Format: Entrada: ... / Saída: ..."""
    pairs = []
    with open(filename, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('Entrada:'):
            entrada = line[len('Entrada:'):].strip()
            if i + 1 < len(lines):
                saida_line = lines[i + 1].strip()
                if saida_line.startswith('Saída:'):
                    saida = saida_line[len('Saída:'):].strip()
                    # Normalize input to lowercase
                    pairs.append((entrada.lower(), saida))
                    i += 2
                    continue
        i += 1

    return pairs


# Load
all_pairs = load_dataset(dataset_file)
print(f'📊 Total pairs loaded: {len(all_pairs):,}')

# Show samples
print('\n📋 Samples:')
for i in range(8):
    src, trg = all_pairs[i]
    print(f'  {i+1}. "{src}" → "{trg}"')

In [ ]:
# Build vocabularies
enc_texts = [pair[0] for pair in all_pairs]  # Portuguese commands
dec_texts = [pair[1] for pair in all_pairs]   # LBML codes

enc_vocab = Vocabulary('encoder')
enc_vocab.build_from_texts(enc_texts)

dec_vocab = Vocabulary('decoder')
dec_vocab.build_from_texts(dec_texts)

print(f'📊 Encoder vocabulary: {enc_vocab.size} tokens')
print(f'   Characters: {sorted([c for c in enc_vocab.stoi if c not in SPECIAL_TOKENS])}')
print(f'\n📊 Decoder vocabulary: {dec_vocab.size} tokens')
print(f'   Characters: {sorted([c for c in dec_vocab.stoi if c not in SPECIAL_TOKENS])}')

# Test encode/decode
test_src = 'vá 40 centímetros para frente'
test_trg = 'D40F;'
print(f'\n🧪 Test encode/decode:')
print(f'   Encoder: "{test_src}" → {enc_vocab.encode(test_src)}')
print(f'   Decoder: "{test_trg}" → {dec_vocab.encode(test_trg)}')
print(f'   Decode back: {dec_vocab.decode(dec_vocab.encode(test_trg))}')

In [ ]:
# Split dataset: 85% train, 10% val, 5% test (BY EXAMPLE)
random.seed(42)
indices = list(range(len(all_pairs)))
random.shuffle(indices)

n_total = len(all_pairs)
n_train = int(n_total * 0.85)
n_val = int(n_total * 0.10)
n_test = n_total - n_train - n_val

train_pairs = [all_pairs[i] for i in indices[:n_train]]
val_pairs = [all_pairs[i] for i in indices[n_train:n_train + n_val]]
test_pairs = [all_pairs[i] for i in indices[n_train + n_val:]]

print(f'📊 Dataset split:')
print(f'   Train: {len(train_pairs):,} ({len(train_pairs)/n_total*100:.1f}%)')
print(f'   Val:   {len(val_pairs):,} ({len(val_pairs)/n_total*100:.1f}%)')
print(f'   Test:  {len(test_pairs):,} ({len(test_pairs)/n_total*100:.1f}%)')

## 4. Dataset e DataLoader

In [ ]:
class LBotDataset(Dataset):
    """Dataset for Seq2Seq training."""

    def __init__(self, pairs, enc_vocab, dec_vocab):
        self.pairs = pairs
        self.enc_vocab = enc_vocab
        self.dec_vocab = dec_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, trg_text = self.pairs[idx]

        # Encode source (Portuguese)
        src_ids = self.enc_vocab.encode(src_text)

        # Encode target (LBML) with SOS and EOS
        trg_ids = [SOS_IDX] + self.dec_vocab.encode(trg_text) + [EOS_IDX]

        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(trg_ids, dtype=torch.long),
        )


def collate_fn(batch):
    """Pad sequences in a batch."""
    src_batch, trg_batch = zip(*batch)

    # Pad source sequences
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    # Pad target sequences
    trg_padded = pad_sequence(trg_batch, batch_first=True, padding_value=PAD_IDX)

    return src_padded, trg_padded


# Create datasets
train_dataset = LBotDataset(train_pairs, enc_vocab, dec_vocab)
val_dataset = LBotDataset(val_pairs, enc_vocab, dec_vocab)
test_dataset = LBotDataset(test_pairs, enc_vocab, dec_vocab)

# Create dataloaders - V7 uses batch_size=128 (V6 used 64)
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

print(f'✅ DataLoaders created')
print(f'   Batch size: {BATCH_SIZE} (V6 used 64)')
print(f'   Train batches: {len(train_loader):,}')
print(f'   Val batches: {len(val_loader):,}')

# Test one batch
src_sample, trg_sample = next(iter(train_loader))
print(f'\n🧪 Sample batch shapes:')
print(f'   Source: {src_sample.shape}')
print(f'   Target: {trg_sample.shape}')

## 5. Definição do Modelo Seq2Seq

In [ ]:
@dataclass
class Seq2SeqConfig:
    """Configuration for the Seq2Seq model."""
    enc_vocab_size: int = 100    # Larger for augmented data (V6 had 80)
    dec_vocab_size: int = 20
    enc_emb_dim: int = 128
    dec_emb_dim: int = 64
    hidden_dim: int = 256
    enc_layers: int = 2
    dec_layers: int = 2
    dropout: float = 0.2
    max_enc_len: int = 250       # V6 had 200; increased for number words
    max_dec_len: int = 100


class Encoder(nn.Module):
    """Bidirectional GRU encoder."""

    def __init__(self, config):
        super().__init__()
        self.hidden_dim = config.hidden_dim
        self.n_layers = config.enc_layers

        self.embedding = nn.Embedding(
            config.enc_vocab_size, config.enc_emb_dim, padding_idx=PAD_IDX
        )
        self.gru = nn.GRU(
            config.enc_emb_dim, config.hidden_dim,
            num_layers=config.enc_layers,
            dropout=config.dropout if config.enc_layers > 1 else 0,
            bidirectional=True, batch_first=True,
        )
        self.fc_hidden = nn.Linear(config.hidden_dim * 2, config.hidden_dim)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.gru(embedded)
        # Combine bidirectional hidden states
        hidden = hidden.view(self.n_layers, 2, -1, self.hidden_dim)
        hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)
        hidden = torch.tanh(self.fc_hidden(hidden))
        return outputs, hidden


class BahdanauAttention(nn.Module):
    """Bahdanau (additive) attention."""

    def __init__(self, config):
        super().__init__()
        self.W_enc = nn.Linear(config.hidden_dim * 2, config.hidden_dim, bias=False)
        self.W_dec = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
        self.V = nn.Linear(config.hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        dec_proj = self.W_dec(decoder_hidden.unsqueeze(1))
        enc_proj = self.W_enc(encoder_outputs)
        energy = torch.tanh(enc_proj + dec_proj)
        attention = self.V(energy).squeeze(2)
        if mask is not None:
            attention = attention.masked_fill(mask, float('-inf'))
        attn_weights = torch.softmax(attention, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn_weights


class Decoder(nn.Module):
    """GRU decoder with Bahdanau attention."""

    def __init__(self, config):
        super().__init__()
        self.hidden_dim = config.hidden_dim
        self.n_layers = config.dec_layers

        self.embedding = nn.Embedding(
            config.dec_vocab_size, config.dec_emb_dim, padding_idx=PAD_IDX
        )
        self.attention = BahdanauAttention(config)
        self.gru = nn.GRU(
            config.dec_emb_dim + config.hidden_dim * 2, config.hidden_dim,
            num_layers=config.dec_layers,
            dropout=config.dropout if config.dec_layers > 1 else 0,
            batch_first=True,
        )
        self.fc_out = nn.Linear(
            config.hidden_dim + config.hidden_dim * 2 + config.dec_emb_dim,
            config.dec_vocab_size,
        )
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_token, hidden, encoder_outputs, mask=None):
        input_token = input_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_token))
        context, attn_weights = self.attention(hidden[-1], encoder_outputs, mask)
        gru_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)
        output, hidden = self.gru(gru_input, hidden)
        output = output.squeeze(1)
        prediction = self.fc_out(
            torch.cat([output, context, embedded.squeeze(1)], dim=1)
        )
        return prediction, hidden, attn_weights


class Seq2Seq(nn.Module):
    """Full Seq2Seq model."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)

    def create_mask(self, src):
        return src == PAD_IDX

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg.size(1)
        dec_vocab_size = self.config.dec_vocab_size

        outputs = torch.zeros(batch_size, trg_len - 1, dec_vocab_size, device=src.device)
        encoder_outputs, hidden = self.encoder(src)
        mask = self.create_mask(src)
        input_token = trg[:, 0]

        for t in range(1, trg_len):
            prediction, hidden, _ = self.decoder(
                input_token, hidden, encoder_outputs, mask
            )
            outputs[:, t - 1, :] = prediction

            if self.training and torch.rand(1).item() < teacher_forcing_ratio:
                input_token = trg[:, t]
            else:
                input_token = prediction.argmax(dim=1)

        return outputs

    @torch.no_grad()
    def translate(self, src, max_len=100):
        self.eval()
        encoder_outputs, hidden = self.encoder(src)
        mask = self.create_mask(src)
        input_token = torch.tensor([SOS_IDX], device=src.device)
        output_tokens = []
        attention_weights = []

        for _ in range(max_len):
            prediction, hidden, attn_weights = self.decoder(
                input_token, hidden, encoder_outputs, mask
            )
            attention_weights.append(attn_weights.cpu())
            top_token = prediction.argmax(dim=1)
            token_id = top_token.item()
            if token_id == EOS_IDX:
                break
            output_tokens.append(token_id)
            input_token = top_token

        return output_tokens, attention_weights


# Create model
config = Seq2SeqConfig(
    enc_vocab_size=enc_vocab.size,
    dec_vocab_size=dec_vocab.size,
    enc_emb_dim=128,
    dec_emb_dim=64,
    hidden_dim=256,
    enc_layers=2,
    dec_layers=2,
    dropout=0.2,
    max_enc_len=250,   # V7: increased from 200
)

model = Seq2Seq(config).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Model created')
print(f'   Total parameters: {total_params:,}')
print(f'   Trainable: {trainable_params:,}')
print(f'   Encoder vocab: {config.enc_vocab_size}')
print(f'   Decoder vocab: {config.dec_vocab_size}')
print(f'   Hidden dim: {config.hidden_dim}')
print(f'   Max enc len: {config.max_enc_len} (V6 had 200)')
print(f'   Encoder: BiGRU {config.enc_layers} layers')
print(f'   Decoder: GRU {config.dec_layers} layers + Bahdanau Attention')
print(f'\n📐 Model architecture:')
print(model)

## 6. Training Loop

In [ ]:
# Hyperparameters — V7 adjustments
LEARNING_RATE = 3e-4
MAX_EPOCHS = 150        # V6 had 100
WARMUP_STEPS = 500
GRAD_CLIP = 1.0
PATIENCE = 15           # V6 had 10
TF_START = 0.5          # Teacher forcing start ratio
TF_END = 0.0            # Teacher forcing end ratio

# Optimizer and loss
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Cosine annealing LR scheduler with warmup
total_steps = MAX_EPOCHS * len(train_loader)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f'✅ Training setup:')
print(f'   Learning rate: {LEARNING_RATE}')
print(f'   Max epochs: {MAX_EPOCHS} (V6 had 100)')
print(f'   Batch size: {BATCH_SIZE} (V6 had 64)')
print(f'   Warmup steps: {WARMUP_STEPS}')
print(f'   Gradient clip: {GRAD_CLIP}')
print(f'   Early stopping patience: {PATIENCE} (V6 had 10)')
print(f'   Teacher forcing: {TF_START} → {TF_END}')
print(f'   Total steps: {total_steps:,}')

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scheduler, tf_ratio, grad_clip, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    n_batches = 0

    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()

        outputs = model(src, trg, teacher_forcing_ratio=tf_ratio)

        output_dim = outputs.shape[-1]
        outputs_flat = outputs.reshape(-1, output_dim)
        targets_flat = trg[:, 1:].reshape(-1)

        loss = criterion(outputs_flat, targets_flat)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate on validation/test set."""
    model.eval()
    total_loss = 0
    n_batches = 0

    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)

        outputs = model(src, trg, teacher_forcing_ratio=0.0)

        output_dim = outputs.shape[-1]
        outputs_flat = outputs.reshape(-1, output_dim)
        targets_flat = trg[:, 1:].reshape(-1)

        loss = criterion(outputs_flat, targets_flat)
        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


@torch.no_grad()
def compute_accuracy(model, pairs, enc_vocab, dec_vocab, device, max_samples=None):
    """Compute exact match accuracy on a set of (input, expected_output) pairs."""
    model.eval()
    correct = 0
    total = 0
    errors = []

    samples = pairs if max_samples is None else pairs[:max_samples]

    for src_text, expected in samples:
        src_ids = enc_vocab.encode(src_text)
        src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)

        output_tokens, _ = model.translate(src_tensor, max_len=100)
        predicted = dec_vocab.decode(output_tokens)

        if predicted == expected:
            correct += 1
        else:
            if len(errors) < 10:
                errors.append((src_text, expected, predicted))

        total += 1

    accuracy = correct / total if total > 0 else 0
    return accuracy, correct, total, errors


print('✅ Training functions defined')

In [ ]:
# ============================================================================
# MAIN TRAINING LOOP
# ============================================================================

print('🚀 Starting training...')
print('=' * 70)

train_losses = []
val_losses = []
val_accuracies = []
best_val_loss = float('inf')
best_val_acc = 0.0
patience_counter = 0
best_epoch = 0

start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()

    # Teacher forcing ratio: linear decay from TF_START to TF_END
    tf_ratio = TF_START - (TF_START - TF_END) * (epoch - 1) / max(1, MAX_EPOCHS - 1)

    # Train
    train_loss = train_epoch(
        model, train_loader, optimizer, criterion, scheduler,
        tf_ratio, GRAD_CLIP, device
    )
    train_losses.append(train_loss)

    # Validate
    val_loss = evaluate(model, val_loader, criterion, device)
    val_losses.append(val_loss)

    # Accuracy check every 5 epochs (on subset for speed)
    val_acc = 0.0
    if epoch % 5 == 0 or epoch == 1:
        val_acc, correct, total, errors = compute_accuracy(
            model, val_pairs, enc_vocab, dec_vocab, device, max_samples=500
        )
        val_accuracies.append((epoch, val_acc))

    epoch_time = time.time() - epoch_start
    current_lr = scheduler.get_last_lr()[0]

    # Log
    acc_str = f' | Val Acc: {val_acc:.1%}' if val_acc > 0 else ''
    print(
        f'Epoch {epoch:3d}/{MAX_EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}{acc_str} | '
        f'TF: {tf_ratio:.2f} | LR: {current_lr:.2e} | Time: {epoch_time:.1f}s'
    )

    # Show errors periodically
    if epoch % 10 == 0 and errors:
        print(f'  Sample errors:')
        for src, exp, pred in errors[:3]:
            print(f'    "{src}" → expected "{exp}", got "{pred}"')

    # Save best model (by val loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        checkpoint = {
            'model': model.state_dict(),
            'config': config,
            'enc_stoi': enc_vocab.stoi,
            'enc_itos': enc_vocab.itos,
            'dec_stoi': dec_vocab.stoi,
            'dec_itos': dec_vocab.itos,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'version': 'v7-seq2seq-robust',
        }
        torch.save(checkpoint, 'lbot_translator_v7.pt')
        print(f'  💾 Saved best model (val_loss={val_loss:.4f})')
    else:
        patience_counter += 1

    # Track best accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc

    # Early stopping
    if patience_counter >= PATIENCE:
        print(f'\n⏹️  Early stopping at epoch {epoch} (patience={PATIENCE})')
        print(f'   Best val_loss: {best_val_loss:.4f} at epoch {best_epoch}')
        break

total_time = time.time() - start_time
print(f'\n✅ Training complete in {total_time/60:.1f} minutes')
print(f'   Best val loss: {best_val_loss:.4f} at epoch {best_epoch}')
print(f'   Best val accuracy: {best_val_acc:.1%}')

## 7. Visualizar Resultados

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses, label='Train Loss', alpha=0.8)
axes[0].plot(val_losses, label='Val Loss', alpha=0.8)
axes[0].axvline(x=best_epoch-1, color='r', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss (V7)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
if val_accuracies:
    epochs_acc, accs = zip(*val_accuracies)
    axes[1].plot(epochs_acc, [a*100 for a in accs], 'go-', label='Val Accuracy', markersize=6)
    axes[1].axhline(y=90, color='r', linestyle='--', alpha=0.5, label='Target: 90%')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Validation Accuracy (exact match)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig('training_curves_v7.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Training curves saved to training_curves_v7.png')

## 8. Avaliação Final

In [ ]:
# Load best model
print('📥 Loading best model...')
checkpoint = torch.load('lbot_translator_v7.pt', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model'])
model.eval()
print(f'   Loaded from epoch {checkpoint["epoch"]}, val_loss={checkpoint["val_loss"]:.4f}')

# Full validation accuracy
print('\n📊 Full validation accuracy...')
val_acc, val_correct, val_total, val_errors = compute_accuracy(
    model, val_pairs, enc_vocab, dec_vocab, device
)
print(f'   Validation Accuracy: {val_acc:.2%} ({val_correct}/{val_total})')

# Full test accuracy
print('\n📊 Test accuracy...')
test_acc, test_correct, test_total, test_errors = compute_accuracy(
    model, test_pairs, enc_vocab, dec_vocab, device
)
print(f'   Test Accuracy: {test_acc:.2%} ({test_correct}/{test_total})')

# Categorized accuracy
print('\n📊 Accuracy by category:')
categories = {
    'Simple Displacement': [],
    'Simple Rotation': [],
    'Compound 2-action': [],
    'Compound 3-action': [],
    'Compound 4-action': [],
}

for src, trg in test_pairs:
    n_actions = trg.count(';')
    if n_actions == 1:
        if trg.startswith('D'):
            categories['Simple Displacement'].append((src, trg))
        else:
            categories['Simple Rotation'].append((src, trg))
    elif n_actions == 2:
        categories['Compound 2-action'].append((src, trg))
    elif n_actions == 3:
        categories['Compound 3-action'].append((src, trg))
    else:
        categories['Compound 4-action'].append((src, trg))

for cat_name, cat_pairs in categories.items():
    if cat_pairs:
        acc, correct, total, _ = compute_accuracy(
            model, cat_pairs, enc_vocab, dec_vocab, device
        )
        print(f'   {cat_name}: {acc:.1%} ({correct}/{total})')

# Show sample errors
if test_errors:
    print(f'\n❌ Sample test errors:')
    for src, exp, pred in test_errors[:10]:
        print(f'   "{src}"')
        print(f'     Expected: {exp}')
        print(f'     Got:      {pred}')
        print()

## 9. Tradução Interativa e Testes

In [ ]:
def translate_command(command, model, enc_vocab, dec_vocab, device):
    """Translate a single command and show result."""
    input_text = command.strip().lower()
    src_ids = enc_vocab.encode(input_text)
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)

    output_tokens, attn_weights = model.translate(src_tensor, max_len=100)
    result = dec_vocab.decode(output_tokens)

    # Validate
    pattern = r'^(D\d+[FBLR];|R\d+[LR];)+$'
    valid = '✅' if re.match(pattern, result) else '❌'

    print(f'  Input:  "{command}"')
    print(f'  Output: "{result}" {valid}')
    return result, attn_weights


# Test cases — V7 includes standard, multi-unit, augmented, and informal inputs
print('🧪 Translation Tests:')
print('=' * 60)

test_commands = [
    # ── Standard (clean, small values) ──
    'vá 40 centímetros para frente',
    'ande 25 centímetros para trás',
    'mova-se 60 centímetros para esquerda',
    'gire 90 graus para direita',
    'vire 45 graus sentido anti-horário',

    # ── 3-digit values (expanded range) ──
    'ande 350 centímetros para frente',
    'ande 999 centímetros para trás',
    'ande 500 centímetros para esquerda',
    'gire 270 graus para esquerda',
    'gire 360 graus para direita',
    'gire 120 graus para direita',

    # ── Multi-unit (preprocessed to cm, clamped if > 999) ──
    'ande 2 metros para frente',
    'mova-se 500 milímetros para trás',
    'avance 3 passos para frente',
    'ande 1 jarda para direita',
    'ande 10 metros para frente',   # should clamp to 999

    # ── Abbreviations ──
    'ande 40cm para frente',
    'gire 90° para direita',
    'ande 350cm para trás',

    # ── Missing accents ──
    'ande 40 centimetros para frente',
    'gire 90 graus sentido anti-horario',

    # ── Numbers as words ──
    'ande quarenta centímetros para frente',
    'gire noventa graus para direita',
    'vá vinte e cinco centímetros para trás',
    'ande trezentos e cinquenta centímetros para frente',

    # ── Missing punctuation ──
    'ande 40 centímetros para frente depois gire 90 graus para direita',

    # ── Informal ──
    'vai 40 centímetros pra frente',
    'roda 90 graus pra esquerda',

    # ── Compound ──
    'vá 40 centímetros para frente e depois gire 90 graus para direita',
    'gire 90 graus para direita e depois vá 30 centímetros para frente',

    # ── Combined augmentations ──
    'vai 40cm pra frente',
    'ande 2 metros pra frente depois roda 90° pra esquerda',
]

for cmd in test_commands:
    translate_command(cmd, model, enc_vocab, dec_vocab, device)
    print()

## 10. Visualizar Attention

In [ ]:
def plot_attention(command, model, enc_vocab, dec_vocab, device):
    """Plot attention heatmap for a command."""
    input_text = command.strip().lower()
    src_ids = enc_vocab.encode(input_text)
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)

    output_tokens, attn_weights = model.translate(src_tensor, max_len=100)
    result = dec_vocab.decode(output_tokens)

    if not attn_weights:
        print('No attention weights to plot')
        return

    # Stack attention weights: [dec_len, src_len]
    attn_matrix = torch.cat(attn_weights, dim=0).squeeze(1).numpy()

    # Labels
    src_chars = list(input_text)
    trg_chars = list(result)

    fig, ax = plt.subplots(figsize=(max(10, len(src_chars) * 0.4), max(4, len(trg_chars) * 0.5)))
    im = ax.imshow(attn_matrix[:len(trg_chars), :len(src_chars)], cmap='YlOrRd', aspect='auto')

    ax.set_xticks(range(len(src_chars)))
    ax.set_xticklabels(src_chars, fontsize=8)
    ax.set_yticks(range(len(trg_chars)))
    ax.set_yticklabels(trg_chars, fontsize=10, fontweight='bold')

    ax.set_xlabel('Input (Portuguese)')
    ax.set_ylabel('Output (LBML)')
    ax.set_title(f'Attention: "{command}" → "{result}"')

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


# Plot attention for key examples
print('🔍 Attention Visualizations:')
print('=' * 60)

attention_examples = [
    'vá 55 centímetros para frente',
    'gire 90 graus para direita',
    'ande 30 centímetros para trás e depois gire 45 graus para esquerda',
    'ande 350 centímetros para frente',       # 3-digit value
    'gire 270 graus para esquerda',           # 3-digit angle
    'vai quarenta centímetros pra frente',    # informal + number words
]

for cmd in attention_examples:
    plot_attention(cmd, model, enc_vocab, dec_vocab, device)

## 11. Benchmark contra Test Set

In [ ]:
# Upload benchmark test set
print('📁 Upload benchmark_test_set.txt')
uploaded = files.upload()
benchmark_file = list(uploaded.keys())[0]
print(f'✅ Uploaded: {benchmark_file}')

In [ ]:
def load_benchmark(filename):
    """Load benchmark test set."""
    tests = []
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if 'Entrada:' in line and '| Saída:' in line:
                parts = line.split('|')
                entrada = parts[0].replace('Entrada:', '').strip()
                saida = parts[1].replace('Saída:', '').strip()
                tests.append((entrada, saida))
    return tests


def run_benchmark(model, tests, enc_vocab, dec_vocab, device):
    """Run benchmark and return detailed results."""
    results = {
        'total': 0, 'correct': 0,
        'simple_disp': {'total': 0, 'correct': 0},
        'simple_rot': {'total': 0, 'correct': 0},
        'compound_2': {'total': 0, 'correct': 0},
        'compound_3': {'total': 0, 'correct': 0},
    }
    errors = []

    model.eval()
    for entrada, expected in tqdm(tests, desc='Benchmarking'):
        input_text = entrada.strip().lower()
        src_ids = enc_vocab.encode(input_text)
        src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)

        output_tokens, _ = model.translate(src_tensor, max_len=100)
        predicted = dec_vocab.decode(output_tokens)

        n_actions = expected.count(';')
        if n_actions == 1:
            cat = 'simple_disp' if expected.startswith('D') else 'simple_rot'
        elif n_actions == 2:
            cat = 'compound_2'
        else:
            cat = 'compound_3'

        results['total'] += 1
        results[cat]['total'] += 1

        if predicted == expected:
            results['correct'] += 1
            results[cat]['correct'] += 1
        else:
            errors.append((entrada, expected, predicted, cat))

    return results, errors


# Run benchmark
benchmark_tests = load_benchmark(benchmark_file)
print(f'📊 Benchmark test set: {len(benchmark_tests)} test cases')

results, errors = run_benchmark(model, benchmark_tests, enc_vocab, dec_vocab, device)

# Print results
print(f'\n{"="*60}')
print(f'📊 BENCHMARK RESULTS - LBot V7 (Seq2Seq + Augmentation)')
print(f'{"="*60}')

overall_acc = results['correct'] / results['total'] * 100
print(f'\n🎯 Overall Accuracy: {overall_acc:.1f}% ({results["correct"]}/{results["total"]})')

categories_display = [
    ('Simple Displacement', 'simple_disp'),
    ('Simple Rotation', 'simple_rot'),
    ('Compound 2-action', 'compound_2'),
    ('Compound 3-action', 'compound_3'),
]

print(f'\n📊 By Category:')
for display_name, key in categories_display:
    cat = results[key]
    if cat['total'] > 0:
        acc = cat['correct'] / cat['total'] * 100
        print(f'   {display_name:25s}: {acc:5.1f}% ({cat["correct"]}/{cat["total"]})')

# Compare with V5 and V6
print(f'\n📊 Comparison with previous versions:')
print(f'   {"Category":25s} {"V5":>8s} {"V6":>8s} {"V7":>8s}')
print(f'   {"-"*55}')
v5_results = {'overall': 61.1, 'simple_rot': 100.0, 'simple_disp': 44.7, 'compound_2': 28.3, 'compound_3': 32.5}
v7_cats = {'overall': overall_acc}
for _, key in categories_display:
    cat = results[key]
    v7_cats[key] = cat['correct'] / cat['total'] * 100 if cat['total'] > 0 else 0

for display_name, key in [('Overall', 'overall')] + categories_display:
    v5_val = v5_results.get(key, 0)
    v7_val = v7_cats.get(key, 0)
    print(f'   {display_name:25s} {v5_val:7.1f}% {"TBD":>8s} {v7_val:7.1f}%')

# Show errors
if errors:
    print(f'\n❌ Errors ({len(errors)} total):')
    for entrada, expected, predicted, cat in errors[:20]:
        print(f'   [{cat}] "{entrada}"')
        print(f'     Expected: {expected}')
        print(f'     Got:      {predicted}')
        print()

## 12. Exportar Modelo

In [ ]:
# Final save with all metadata
final_checkpoint = {
    'model': model.state_dict(),
    'config': config,
    'enc_stoi': enc_vocab.stoi,
    'enc_itos': enc_vocab.itos,
    'dec_stoi': dec_vocab.stoi,
    'dec_itos': dec_vocab.itos,
    'version': 'v7-seq2seq-robust',
    'train_losses': train_losses,
    'val_losses': val_losses,
    'val_accuracies': val_accuracies,
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'benchmark_results': results if 'results' in dir() else None,
}
torch.save(final_checkpoint, 'lbot_translator_v7.pt')

# Download
print(f'📦 Model saved: lbot_translator_v7.pt')
print(f'   Size: {os.path.getsize("lbot_translator_v7.pt") / 1024 / 1024:.1f} MB')

files.download('lbot_translator_v7.pt')
files.download('training_curves_v7.png')

print('\n✅ Export complete! Files downloaded:')
print('   • lbot_translator_v7.pt  (model checkpoint)')
print('   • training_curves_v7.png (training visualization)')

## 13. Interface Interativa

In [ ]:
import unicodedata

# ── V7 Preprocessing Pipeline (same as lbot_v7.py) ──

_UNITS_WORDS_MAP = {
    'zero': 0, 'um': 1, 'uma': 1, 'dois': 2, 'duas': 2, 'três': 3, 'tres': 3,
    'quatro': 4, 'cinco': 5, 'seis': 6, 'sete': 7, 'oito': 8, 'nove': 9,
    'dez': 10, 'onze': 11, 'doze': 12, 'treze': 13, 'quatorze': 14, 'catorze': 14,
    'quinze': 15, 'dezesseis': 16, 'dezessete': 17, 'dezoito': 18, 'dezenove': 19,
}
_TENS_MAP = {
    'vinte': 20, 'trinta': 30, 'quarenta': 40, 'cinquenta': 50,
    'cinqüenta': 50, 'sessenta': 60, 'setenta': 70, 'oitenta': 80, 'noventa': 90,
}
_HUNDREDS_MAP = {
    'cem': 100, 'cento': 100, 'duzentos': 200, 'duzentas': 200,
    'trezentos': 300, 'trezentas': 300, 'quatrocentos': 400, 'quatrocentas': 400,
    'quinhentos': 500, 'quinhentas': 500, 'seiscentos': 600, 'seiscentas': 600,
    'setecentos': 700, 'setecentas': 700, 'oitocentos': 800, 'oitocentas': 800,
    'novecentos': 900, 'novecentas': 900,
}
_ALL_NUM_WORDS = set(_UNITS_WORDS_MAP) | set(_TENS_MAP) | set(_HUNDREDS_MAP) | {'e'}

def _words_to_num(words):
    filtered = [w for w in words if w != 'e']
    if not filtered: return None
    total = 0
    for w in filtered:
        if w in _HUNDREDS_MAP: total += _HUNDREDS_MAP[w]
        elif w in _TENS_MAP: total += _TENS_MAP[w]
        elif w in _UNITS_WORDS_MAP: total += _UNITS_WORDS_MAP[w]
        else: return None
    return total if total > 0 or (len(filtered) == 1 and filtered[0] == 'zero') else None

def preprocess_input(text):
    """V7 preprocessing pipeline."""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    
    # Informal normalization
    informal_replacements = [
        (r'\bvai\b', 'vá'), (r'\bsegue\b', 'siga'), (r'\bmexe\b', 'mova-se'),
        (r'\bbota pra andar\b', 'ande'), (r'\bse mexe\b', 'se mova'),
        (r'\broda\b', 'rode'), (r'\bdobra\b', 'gire'),
        (r'\bfaz uma curva\b', 'faça uma curva'),
        (r'\bmuda de dire[çc][ãa]o\b', 'mude a direção'),
        (r'\breto\b', 'para frente'), (r'\bde r[ée]\b', 'para trás'),
        (r'\bde costas\b', 'para trás'),
        (r'\buns\s+(\d)', r'\1'),
        (r'\baí\s+depois\b', 'depois'), (r'\bai\s+depois\b', 'depois'),
        (r'\be\s+aí\b', 'e depois'), (r'\be\s+ai\b', 'e depois'),
        (r'\bdaí\b', 'depois'), (r'\bdai\b', 'depois'),
    ]
    for pat, rep in informal_replacements:
        text = re.sub(pat, rep, text, flags=re.IGNORECASE)
    
    # Number words -> digits
    words = text.split()
    result = []; i = 0
    while i < len(words):
        wl = words[i].lower()
        if wl in _ALL_NUM_WORDS and wl != 'e':
            num_words = []; j = i
            while j < len(words) and words[j].lower() in _ALL_NUM_WORDS:
                num_words.append(words[j].lower()); j += 1
            number = _words_to_num(num_words)
            if number is not None:
                result.append(str(number)); i = j; continue
        result.append(words[i]); i += 1
    text = ' '.join(result)
    
    # Unit conversion
    unit_patterns = [
        (r'(\d+(?:[.,]\d+)?)\s*(?:quil[oô]metros?|quilometros?|km)\b', 100000),
        (r'(\d+(?:[.,]\d+)?)\s*(?:mil[ií]metros?|milimetros?|mm)\b', 0.1),
        (r'(\d+(?:[.,]\d+)?)\s*(?:jardas?|yd)\b', 91.44),
        (r'(\d+(?:[.,]\d+)?)\s*(?:passos?)\b', 75),
        (r'(\d+(?:[.,]\d+)?)\s*(?:metros?|m)\b(?!\w)', 100),
    ]
    for pattern, factor in unit_patterns:
        def _repl(match, f=factor):
            v = float(match.group(1).replace(',', '.'))
            cm = round(v * f)
            u = 'centímetro' if cm == 1 else 'centímetros'
            return f'{cm} {u}'
        text = re.sub(pattern, _repl, text, flags=re.IGNORECASE)
    
    # Abbreviations
    text = re.sub(r'(\d+)\s*cm\b', r'\1 centímetros', text, flags=re.IGNORECASE)
    text = re.sub(r'(\d+)\s*°', r'\1 graus', text)
    
    # Accent fixes
    accent_fixes = {
        'centimetros': 'centímetros', 'centimetro': 'centímetro',
        'horario': 'horário', 'anti-horario': 'anti-horário',
        'atras': 'atrás', 'a frente': 'à frente',
        'a esquerda': 'à esquerda', 'a direita': 'à direita',
    }
    for wrong, correct in accent_fixes.items():
        text = re.sub(r'\b' + re.escape(wrong) + r'\b', correct, text)
    
    # Punctuation normalization
    connectors = ['depois', 'em seguida', 'então', 'entao', 'por fim',
                  'aí depois', 'ai depois', 'e aí', 'e ai', 'daí', 'dai']
    for conn in sorted(connectors, key=len, reverse=True):
        pat = r'(?<![,;.]\s?)(?<![,;.])\s+(' + re.escape(conn) + r')\b'
        text = re.sub(pat, r', \1', text, flags=re.IGNORECASE)
    
    return text


def interactive_translator_v7(model, enc_vocab, dec_vocab, device):
    """Interactive translation REPL with V7 preprocessing."""
    print('\n🤖 === LBOT V7 TRANSLATOR (Seq2Seq + Preprocessing) ===')
    print('Digite comandos em português ou "sair" para terminar')
    print('Agora aceita: abreviações (cm, °), sem acentos, metros, passos, informal, etc.')
    print()

    model.eval()
    while True:
        try:
            command = input('🗣️  Comando: ').strip()
            if command.lower() in ['sair', 'exit', 'quit', '']:
                print('👋 Tchau!')
                break

            preprocessed = preprocess_input(command)
            input_text = preprocessed.lower()
            
            if command.strip().lower() != input_text:
                print(f'🔧 Preprocessed: {preprocessed}')
            
            src_ids = enc_vocab.encode(input_text)
            src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)
            output_tokens, _ = model.translate(src_tensor, max_len=100)
            result = dec_vocab.decode(output_tokens)

            pattern = r'^(D\d+[FBLR];|R\d+[LR];)+$'
            valid = '✅' if re.match(pattern, result) else '⚠️'
            print(f'🤖 LBot: {result} {valid}\n')

        except KeyboardInterrupt:
            print('\n👋 Tchau!')
            break


interactive_translator_v7(model, enc_vocab, dec_vocab, device)